# **Variable Processing** #

Project Title: Identification of Significant Severe Weather Regimes Using Self-Organizing Maps

Authors: Kelvin Hawthorne and Lexi Gutzwiler

Northern Illinois University Department of Earth, Atmosphere, and Environment

**NOTEBOOK 2 FOR PROJECT METHODS**

## **Package Imports** ##

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
from tqdm import tqdm

## **Path Specification** ##

In [ ]:
in_dir = "/glade/derecho/scratch/khawthorne/485 project data"
out_dir = "/glade/derecho/scratch/khawthorne/485 project data/processed_500mb_lapserate"

os.makedirs(out_dir, exist_ok=True)

files = sorted(glob.glob(os.path.join(in_dir, "wrf3d_ECONUS_Z_P_TK_WY*.nc")))

print(f"Found {len(files)} files")

## **Variable Processing** ##

### **Interpolate One Vertical Profile** ###

In [ ]:
def interp_to_pressure(var_prof, p_prof, target_p):
    var_prof = np.asarray(var_prof)
    p_prof = np.asarray(p_prof)

    # remove bad values
    mask = np.isfinite(var_prof) & np.isfinite(p_prof)
    if mask.sum() < 2:
        return np.nan

    var_prof = var_prof[mask]
    p_prof = p_prof[mask]

    # sort pressure ascending for np.interp
    sort_idx = np.argsort(p_prof)
    p_sorted = p_prof[sort_idx]
    var_sorted = var_prof[sort_idx]

    # if target pressure is outside the profile range, return nan
    if (target_p < p_sorted.min()) or (target_p > p_sorted.max()):
        return np.nan

    return np.interp(target_p, p_sorted, var_sorted)

### **Conversions** ###

In [ ]:
for f in tqdm(files, desc="Processing chunk files"):
    ds = xr.open_dataset(f)

   
    ds["TK"] = ds["TK"] - 273.15 #Converting temperature from K to C
    ds["TK"].attrs["units"] = "degC" #Redefining the unit in the dataset
    ds["TK"].attrs["description"] = "Temperature"

 
    vert_dim = None
    for d in ds["P"].dims:
        if d not in ["time", "south_north", "west_east"]:
            vert_dim = d
            break

    if vert_dim is None:
        raise ValueError(f"Could not identify vertical dimension in {f}")

### **Interpolation of Pressure (P) and Geopotential Height (Z) over Pressure Levels** ###

In [ ]:
  z500 = xr.apply_ufunc(
        interp_to_pressure,
        ds["Z"],
        ds["P"],
        input_core_dims=[[vert_dim], [vert_dim]],
        output_core_dims=[[]],
        vectorize=True,
        kwargs={"target_p": 500.0},
        dask="parallelized",
        output_dtypes=[float],
    )

In [ ]:
z700 = xr.apply_ufunc(
        interp_to_pressure,
        ds["Z"],
        ds["P"],
        input_core_dims=[[vert_dim], [vert_dim]],
        output_core_dims=[[]],
        vectorize=True,
        kwargs={"target_p": 700.0},
        dask="parallelized",
        output_dtypes=[float],
    )

### **Interpolation of Pressure (P) and Temperature (TK) over Pressure Levels** ###

In [ ]:
 t500 = xr.apply_ufunc(
        interp_to_pressure,
        ds["TK"],
        ds["P"],
        input_core_dims=[[vert_dim], [vert_dim]],
        output_core_dims=[[]],
        vectorize=True,
        kwargs={"target_p": 500.0},
        dask="parallelized",
        output_dtypes=[float],
    )

    t700 = xr.apply_ufunc(
        interp_to_pressure,
        ds["TK"],
        ds["P"],
        input_core_dims=[[vert_dim], [vert_dim]],
        output_core_dims=[[]],
        vectorize=True,
        kwargs={"target_p": 700.0},
        dask="parallelized",
        output_dtypes=[float],
    )

## **Variable Creation** ##

### **Creation of 500 hPa Geopotential Height Variable** ###

In [ ]:
ds["Z"] = z500
    ds["Z"].attrs["units"] = "m"
    ds["Z"].attrs["description"] = "500 mb geopotential height"

### **Creation of 500-700 hPa Lapse Rate Variable** ###

In [ ]:
 dz_km = (z500 - z700) / 1000.0
    lapse_rate = (t700 - t500) / dz_km

    ds["75 LR"] = lapse_rate
    ds["75 LR"].attrs["units"] = "degC/km"
    ds["75 LR"].attrs["description"] = "700-500 mb lapse rate"

### **Creation of 500 and 700 hPa Temperature Variables** ###

In [ ]:
ds["TK_500mb"] = t500
    ds["TK_500mb"].attrs["units"] = "degC"
    ds["TK_500mb"].attrs["description"] = "500 mb temperature"

    ds["TK_700mb"] = t700
    ds["TK_700mb"].attrs["units"] = "degC"
    ds["TK_700mb"].attrs["description"] = "700 mb temperature"

In [ ]:
 out_path = os.path.join(out_dir, out_name)

    ds.to_netcdf(out_path)
    ds.close()